# 第13回 類似度・クラスタリング・判別

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：2本の系列が「似ている」ことを、どう測ればよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "feasts", "fda", "dtw"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

`fda`、`dtw`、`cluster`、`MASS`の4つのパッケージは、`library()`で読み込まず、`fda::`のようにパッケージの名前を付けて呼びます。
読み込むと、`dplyr`の`select`や、`stats`の`dist`が、別の関数に置き換わるためです。

### A-1. 形で測る：ユークリッド距離とDTW

同じ形の山を、時間の方向に8時点ずらして、2本の系列を作ります。長さはどちらも48です。

In [ ]:
m <- 48
bump <- function(t, centre, width) exp(-(t - centre)^2 / (2 * width^2))   # 山の形
x <- bump(1:m, 20, 4)
y <- bump(1:m, 28, 4)
ggplot(rbind(data.frame(t = 1:m, series = "x", value = x),
             data.frame(t = 1:m, series = "y", value = y)),
       aes(t, value, colour = series)) +
  geom_line() + geom_point(size = 1) +
  labs(x = "Time t (step)", y = "Value", colour = "Series")

ユークリッド距離（Euclidean distance）は、同じ時点どうしを比べます。

$$
d_{\mathrm{E}}(x, y) = \sqrt{\sum_{i=1}^{m} (x_i - y_i)^2}
$$

2本の系列の、どの点とどの点を比べたかを、線で結んで描く関数を用意します。見やすいように、`x`を上にずらして描きます。

In [ ]:
# x の i 番目の点と、y の j 番目の点を、線で結んで描く（i, j は同じ長さの番号の並び）
plot_match <- function(x, y, i, j, offset = 1.5) {
  ggplot() +
    geom_segment(data = data.frame(i = i, j = j, xi = x[i] + offset, yj = y[j]),
                 aes(x = i, xend = j, y = xi, yend = yj), colour = "grey60", linewidth = 0.3) +
    geom_line(data = data.frame(t = seq_along(x), value = x + offset), aes(t, value), colour = "#F8766D") +
    geom_line(data = data.frame(t = seq_along(y), value = y), aes(t, value), colour = "#00BFC4") +
    labs(x = "Time t (step)", y = "Value (x is shifted upward)")
}

In [ ]:
plot_match(x, y, 1:m, 1:m)
sqrt(sum((x - y)^2))

線は、すべて真下に降りています。`x`の山は`y`の平らなところと、`y`の山は`x`の平らなところと比べられています。
形は同じなのに、距離は大きくなります。

動的時間伸縮法（dynamic time warping, DTW）は、時間の方向の伸び縮みを許して、点どうしを結び直します。第12回の最後に触れた、位置合わせ（registration）に近い考え方です。
結び方（道）には、決まりがあります。

- 最初の点どうし、最後の点どうしを結ぶ。
- 順番を入れ替えない。飛ばさない。

決まりを守る結び方のうち、結んだ点どうしの差の2乗の合計が、最も小さいものを探します。その合計の平方根が、DTWの距離です。

探し方は、表を1マスずつ埋める方法（動的計画法）です。$D(i, j)$ を、「`x`の$i$番目までと、`y`の$j$番目までを、最も安く結んだときの合計」とします。

$$
D(i, j) = (x_i - y_j)^2 + \min \{ D(i-1, j),\; D(i, j-1),\; D(i-1, j-1) \}, \qquad d_{\mathrm{DTW}}(x, y) = \sqrt{D(m, m)}
$$

`dtw`パッケージで計算します。

In [ ]:
local_cost <- outer(x, y, function(a, b) (a - b)^2)       # (i, j) のマスが (x_i - y_j)^2 の表
al <- dtw::dtw(local_cost, step.pattern = dtw::symmetric1, keep.internals = TRUE)
plot_match(x, y, al$index1, al$index2)
c(euclid = sqrt(sum((x - y)^2)), dtw = sqrt(al$distance))

- 線が斜めになりました。`x`の山は、`y`の山と結ばれています。
- DTWの距離は、ほぼ0です。「時間の方向にずれただけの、同じ形」と測っています。

表 $D(i, j)$ と、選ばれた道（赤い線）を描きます。

In [ ]:
cost <- data.frame(i = as.vector(row(al$costMatrix)), j = as.vector(col(al$costMatrix)),
                   D = as.vector(al$costMatrix))
ggplot(cost, aes(i, j)) +
  geom_raster(aes(fill = D)) +
  geom_abline(slope = 1, intercept = 0, colour = "white", linetype = "dashed") +
  geom_path(data = data.frame(i = al$index1, j = al$index2), colour = "red", linewidth = 1) +
  scale_fill_viridis_c() +
  coord_fixed() +
  labs(x = "Index i of x", y = "Index j of y", fill = "D(i, j)")

- 白い破線は、対角線（$i = j$）です。対角線をまっすぐ進む道が、ユークリッド距離にあたります。
- 赤い道は、対角線から離れて、色の暗い（安い）ところを通っています。

道が対角線から離れてよい幅を、窓（window）で制限できます。窓の幅を $w$ とすると、$|i - j| \le w$ のマスだけを通ります。

このあとは、たくさんの組の距離を一度に計算します。そのための関数を用意します。上の式を、そのまま書いたものです。

In [ ]:
# 行列 A と B の同じ行どうしのDTWの距離（1行が1本の系列。window は窓の幅）
dtw_pairs <- function(A, B, window = ncol(A)) {
  m <- ncol(A)
  prev <- rep(list(Inf), m)                      # 表の1つ前の行：D(i-1, 1), ..., D(i-1, m)
  for (i in 1:m) {
    cur <- rep(list(Inf), m)                     # 表の今の行：D(i, 1), ..., D(i, m)
    for (j in max(1, i - window):min(m, i + window)) {
      best <- if (i == 1 && j == 1) 0 else
        pmin(prev[[j]], if (j > 1) cur[[j - 1]] else Inf, if (j > 1) prev[[j - 1]] else Inf)
      cur[[j]] <- (A[, i] - B[, j])^2 + best
    }
    prev <- cur
  }
  sqrt(prev[[m]])
}
# 行列 M の行どうしの、すべての組のDTWの距離（dist() と同じ形で返す）
dtw_dist <- function(M, window = ncol(M)) {
  n <- nrow(M)
  pair <- which(lower.tri(diag(n)), arr.ind = TRUE)
  d <- dtw_pairs(M[pair[, 1], , drop = FALSE], M[pair[, 2], , drop = FALSE], window)
  structure(d, Size = n, Diag = FALSE, Upper = FALSE, class = "dist")
}
# 行列 M の各行から、行列 M_ref の各行までのDTWの距離（行が M、列が M_ref の表）
dtw_cross <- function(M, M_ref, window = ncol(M)) {
  pair <- expand.grid(i = seq_len(nrow(M)), j = seq_len(nrow(M_ref)))
  matrix(dtw_pairs(M[pair$i, , drop = FALSE], M_ref[pair$j, , drop = FALSE], window), nrow(M))
}

窓の幅を変えて、`x`と`y`の距離を計算します。

In [ ]:
w <- c(0, 2, 4, 6, 8, 48)
data.frame(window = w,
           distance = sapply(w, function(w) dtw_pairs(rbind(x), rbind(y), w)))

- 窓の幅が0のとき、ユークリッド距離と同じ値になります。
- 窓を広げるほど、距離は小さくなります。ずれ（8時点）と同じ幅まで広げると、ほぼ0になります。
- 窓を制限しないとき（48）の値は、`dtw`パッケージの値と一致します。

### A-2. 距離を取り替えて、クラスタリングする

形の違う3つのグループを作ります。1つのグループは30本です。

- グループ1：山が1つ
- グループ2：山が2つ
- グループ3：山のあとに谷

同じものを2通り作ります。`aligned`は、時間の方向のずれがありません。`shifted`は、1本ごとに、最大で8時点、左右にずらしてあります。

In [ ]:
set.seed(1)
shape <- function(t, g) {
  switch(g, bump(t, 24, 4), bump(t, 18, 3) + bump(t, 30, 3), bump(t, 19, 3.5) - bump(t, 29, 3.5))
}
# 1つのグループが n_each 本。max_shift は、ずれの最大の幅
make_curves <- function(n_each, max_shift, noise_sd = 0.1) {
  group <- rep(1:3, each = n_each)
  shift <- round(runif(length(group), -max_shift, max_shift))
  M <- t(sapply(seq_along(group), function(i) shape(1:m - shift[i], group[i]) + rnorm(m, sd = noise_sd)))
  list(M = M, group = group)
}
aligned <- make_curves(30, 0)
shifted <- make_curves(30, 8)

In [ ]:
# 行列 M の各行を、1本の線として描く。group ごとに枠を分ける
plot_curves <- function(M, group, time = seq_len(ncol(M))) {
  d <- data.frame(id = rep(seq_len(nrow(M)), each = ncol(M)), time = rep(time, nrow(M)),
                  group = rep(group, each = ncol(M)), value = as.numeric(t(M)))
  ggplot(d, aes(time, value, group = id)) +
    geom_line(alpha = 0.3) +
    facet_wrap(~group, nrow = 1) +
    labs(x = "Time t (step)", y = "Value")
}

In [ ]:
plot_curves(rbind(aligned$M, shifted$M),
            paste(rep(c("aligned", "shifted"), each = 90), "group", c(aligned$group, shifted$group))) +
  facet_wrap(~group, nrow = 2)

クラスタリング（clustering）は、ラベルを使わずに、似たものどうしをまとめる方法です。
多変量解析で学んだ方法は、どれも、対象どうしの距離の表（距離行列）があれば使えます。距離を取り替えれば、同じ方法で、まとまり方が変わります。

ここでは、k-medoids法（`cluster`パッケージの`pam`）を使います。k-means法の「クラスターの平均」の代わりに、「クラスターを代表する1本」（メドイド、medoid）を使う方法で、距離行列だけで計算できます。

`shifted`の90本を、ユークリッド距離とDTWで、3つに分けます。

In [ ]:
D_euclid <- dist(shifted$M)             # ユークリッド距離
D_dtw    <- dtw_dist(shifted$M)         # DTW
cl_euclid <- cluster::pam(D_euclid, k = 3)$clustering
cl_dtw    <- cluster::pam(D_dtw, k = 3)$clustering
table(group = shifted$group, cluster = cl_euclid)
table(group = shifted$group, cluster = cl_dtw)

行が本当のグループ、列が見つかったクラスターの、クロス表です。

- DTW：どのグループも、1つのクラスターにまとまっています。
- ユークリッド距離：グループが、複数のクラスターに散らばっています。

2つの分け方がどのくらい一致しているかを、1つの数で測ります。
すべての組（2本ずつ）について、「2つの分け方の両方で、同じ側（どちらでも同じクラスター、または、どちらでも別のクラスター）に入っているか」を数えます。
その割合が、ランド指数（Rand index）です。でたらめに分けても、ランド指数は0になりません。
でたらめに分けたときの期待値が0、完全な一致が1になるように直したものが、調整ランド指数（adjusted Rand index, ARI）です。
クラスターの番号の付け方には、よりません。

In [ ]:
# 調整ランド指数（a, b は、分け方を表すラベルの並び）
ari <- function(a, b) {
  tab <- table(a, b)
  same_both <- sum(choose(tab, 2))                 # 両方で同じクラスターに入る組の数
  same_a <- sum(choose(rowSums(tab), 2))           # a で同じクラスターに入る組の数
  same_b <- sum(choose(colSums(tab), 2))           # b で同じクラスターに入る組の数
  expected <- same_a * same_b / choose(sum(tab), 2)
  (same_both - expected) / ((same_a + same_b) / 2 - expected)
}

2つのデータ（`aligned`と`shifted`）を、3つの距離で分けて、本当のグループとのARIを出します。
3つ目は、窓の幅を3に制限したDTWです。

In [ ]:
ari_shape <- sapply(list(aligned = aligned, shifted = shifted), function(d) {
  D <- list(euclid = dist(d$M), dtw = dtw_dist(d$M), dtw_window3 = dtw_dist(d$M, window = 3))
  sapply(D, function(D) ari(cluster::pam(D, k = 3)$clustering, d$group))
})
round(ari_shape, 2)

- ずれがなければ（`aligned`）、どの距離でも、本当のグループと完全に一致します。
- ずれがあると（`shifted`）、ユークリッド距離は崩れ、DTWは崩れません。
- 窓が、ずれ（最大8時点）より狭いと、DTWでも足りません。

### A-3. 特徴量で測る

系列そのものではなく、系列を要約した数（特徴量、feature）を比べる方法があります。

長さ120の系列を、3つのグループで20本ずつ作ります。周期は12です。

- `trend`：傾向が強い（上がるか下がるかは、1本ごとに違う）
- `seasonal`：季節が強い（山の位置は、1本ごとに違う）
- `noise`：どちらも弱い

In [ ]:
set.seed(1)
len <- 120
make_series <- function(g) {
  slope <- sample(c(-1, 1), 1) * runif(1, 0.02, 0.04)     # 傾きの向きと大きさ
  phase <- runif(1, 0, 2 * pi)                            # 季節の山の位置
  a <- switch(g, trend = c(1, 0.3), seasonal = c(0, 1.5), noise = c(0, 0.3))
  ar1 <- as.numeric(stats::filter(rnorm(len, sd = 0.7), 0.3, method = "recursive"))   # AR(1)のノイズ
  10 + a[1] * slope * (1:len) + a[2] * sin(2 * pi * (1:len) / 12 + phase) + ar1
}
group_f <- rep(c("trend", "seasonal", "noise"), each = 20)
S <- t(sapply(group_f, make_series))
plot_curves(S[c(1:4, 21:24, 41:44), ], group_f[c(1:4, 21:24, 41:44)]) +
  facet_wrap(~group + id, nrow = 3) +
  theme(strip.text = element_blank())

上の段が`noise`、中の段が`seasonal`、下の段が`trend`です（4本ずつ）。同じグループでも、線の向きや、山の位置は、そろっていません。

特徴量には、第1回と第2回の道具を使います。

- 傾向の強さ（strength of trend）：STL分解 $y_t = T_t + S_t + R_t$ で、$F_T = \max\{0,\; 1 - \mathrm{Var}(R_t) / \mathrm{Var}(T_t + R_t)\}$
- 季節の強さ（strength of seasonality）：$F_S = \max\{0,\; 1 - \mathrm{Var}(R_t) / \mathrm{Var}(S_t + R_t)\}$
- ラグ1の標本自己相関 $\hat{\rho}(1)$
- 差分をとった系列の、ラグ1の標本自己相関

どれも、0と1のあいだ（自己相関は $-1$ と1のあいだ）の数で、系列の長さや単位によりません。

`feasts`パッケージの`features()`で、1本ごとに計算します。

In [ ]:
# 行列 M の各行（1本の系列）の特徴量を計算する（period は季節の周期。季節がなければ1）
series_features <- function(M, period = 1) {
  long <- tsibble(id = rep(seq_len(nrow(M)), each = ncol(M)), t = rep(seq_len(ncol(M)), nrow(M)),
                  value = as.numeric(t(M)), key = id, index = t)
  long |> features(value, list(feat_stl, feat_acf), .period = period)
}
# 行ごとに、平均0・標準偏差1にそろえる（標準化）
standardise <- function(M) t(scale(t(M)))

In [ ]:
ft <- series_features(S, period = 12) |>
  select(trend_strength, seasonal_strength_12, acf1)
ggplot(cbind(ft, group = group_f), aes(trend_strength, seasonal_strength_12, colour = group)) +
  geom_point() +
  labs(x = "Strength of trend", y = "Strength of seasonality", colour = "Group")

60本の系列が、それぞれ1つの点になりました。3つのグループは、きれいに分かれています。

特徴量の表は、行が系列、列が変数の、ふつうの多変量データです。多変量解析の方法が、そのまま使えます。
列ごとに標準化してからユークリッド距離を計算し、同じk-medoids法で分けます。比べるために、形による2つの距離（系列ごとに標準化してから計算する）でも分けます。

In [ ]:
Z <- standardise(S)
D <- list(euclid  = dist(Z),
          dtw     = dtw_dist(Z, window = 12),
          feature = dist(scale(ft)))
ari_feature <- sapply(D, function(D) ari(cluster::pam(D, k = 3)$clustering, group_f))
round(ari_feature, 2)

- 特徴量による距離は、本当のグループと一致します。
- 形による距離は、どちらも崩れます。同じグループでも、線の向きや山の位置が違うので、点どうしを比べても近くなりません。

### A-4. 関数として測る：FPCAのスコア

第12回の関数主成分分析（functional principal component analysis, FPCA）は、1本の曲線を、少数のスコアで表しました。

$$
x_i(t) \approx \hat{\mu}(t) + \sum_{j=1}^{K} z_{ij}\, \xi_j(t)
$$

$\hat{\mu}(t)$ は平均関数、$\xi_j(t)$ は固有関数、$z_{ij}$ は曲線 $i$ の第 $j$ 主成分のスコアです（記号は第12回と同じ）。
スコアの組 $(z_{i1}, \dots, z_{iK})$ どうしのユークリッド距離を、曲線どうしの距離として使います。

第12回では、基底関数を50個置き、粗さの罰則の強さをGCVで選びました。ここでは、計算を簡単にするために、基底関数を15個に減らし、罰則は付けません。固有関数の符号も、そろえません。符号を逆にしても、距離と判別の結果は変わらないからです。

In [ ]:
# 行列 M の各行を、Bスプライン基底でなめらかな関数にする
to_fd <- function(M, nbasis = 15) {
  basis <- fda::create.bspline.basis(c(1, ncol(M)), nbasis = nbasis)
  fda::smooth.basis(seq_len(ncol(M)), t(M), basis)$fd
}
# FPCA：平均関数、固有関数（K 個）、スコア、寄与率を返す
fpca_fit <- function(M, K = 3) {
  fd <- to_fd(M)
  pc <- fda::pca.fd(fd, nharm = K)
  list(mean = fda::mean.fd(fd), harmonics = pc$harmonics, scores = pc$scores, varprop = pc$varprop)
}
# 新しい曲線のスコア：平均関数を引き、固有関数との内積をとる
fpca_scores <- function(M, fit) {
  fd <- to_fd(M)
  centred <- fda::fd(fd$coefs - as.numeric(fit$mean$coefs), fd$basis)
  fda::inprod(centred, fit$harmonics)
}

A-2の2つのデータで、スコア（3つ）を計算します。

In [ ]:
fit_aligned <- fpca_fit(aligned$M)
fit_shifted <- fpca_fit(shifted$M)
sc <- rbind(data.frame(data = "aligned", group = factor(aligned$group), fit_aligned$scores),
            data.frame(data = "shifted", group = factor(shifted$group), fit_shifted$scores))
ggplot(sc, aes(X1, X2, colour = group)) +
  geom_point() +
  facet_wrap(~data) +
  labs(x = "Score of PC1", y = "Score of PC2", colour = "Group")
round(rbind(aligned = fit_aligned$varprop, shifted = fit_shifted$varprop), 2)

下の表は、各主成分の寄与率です。スコアによる距離が、ユークリッド距離とどのくらい似ているかを、すべての組の距離の相関で確かめます。あわせて、スコアの距離でクラスタリングします。

In [ ]:
fpca_check <- function(d, fit) {
  D_score <- dist(fit$scores)
  c(cor_with_euclid = cor(as.numeric(dist(d$M)), as.numeric(D_score)),
    ari = ari(cluster::pam(D_score, k = 3)$clustering, d$group))
}
round(rbind(aligned = fpca_check(aligned, fit_aligned),
            shifted = fpca_check(shifted, fit_shifted)), 2)

- スコアによる距離は、ユークリッド距離とほとんど同じ動きをします。48個の数の距離を、3個の数の距離で近似したことになります。
- そのため、ずれのあるデータでは、ユークリッド距離と同じように崩れます。FPCAも、同じ時刻どうしを比べる方法です。

### A-5. 判別：ラベルを当てる

判別（classification）は、ラベルの分かっている系列（訓練データ）を使って、新しい系列のラベルを当てる問題です。
評価は、第2回と同じ考え方です。ラベルを隠した系列（テストデータ）を取り分け、当てて、隠したラベルと比べます。当たった割合が、正解率（accuracy）です。

方法は2つです。

- 最近傍法（nearest neighbour method）：訓練データの中で、いちばん近い系列を探し、そのラベルで当てる。距離を取り替えられる。
- 関数判別（functional discriminant analysis）：FPCAのスコアに、多変量解析で学んだ線形判別分析（linear discriminant analysis, LDA）を使う。

In [ ]:
# 行列 M の各行から、行列 M_ref の各行までのユークリッド距離（行が M、列が M_ref の表）
euclid_cross <- function(M, M_ref) {
  sqrt(pmax(outer(rowSums(M^2), rowSums(M_ref^2), "+") - 2 * M %*% t(M_ref), 0))
}
# 最近傍法：距離の表（行がテスト、列が訓練）と、訓練データのラベルから、テストデータのラベルを当てる
nearest_label <- function(D_cross, label_train) label_train[apply(D_cross, 1, which.min)]
# 関数判別：訓練データでFPCAとLDAを行い、テストデータのラベルを当てる
fpca_lda <- function(M_train, label_train, M_test, K = 3) {
  fit <- fpca_fit(M_train, K)
  lda <- MASS::lda(fit$scores, grouping = label_train)
  as.character(predict(lda, fpca_scores(M_test, fit))$class)
}
# 3つの方法の正解率
accuracy_table <- function(M_train, label_train, M_test, label_test, window = ncol(M_train)) {
  pred <- list(
    nn_euclid = nearest_label(euclid_cross(M_test, M_train), label_train),
    nn_dtw    = nearest_label(dtw_cross(M_test, M_train, window), label_train),
    lda_fpca  = fpca_lda(M_train, label_train, M_test))
  sapply(pred, function(p) mean(p == label_test))
}

訓練データは1つのグループが5本（合わせて15本）、テストデータは1つのグループが50本（合わせて150本）です。
A-2と同じ作り方で、ずれのない場合と、ずれのある場合を比べます。

In [ ]:
set.seed(2)
acc_sim <- sapply(c(aligned = 0, shifted = 8), function(max_shift) {
  train <- make_curves(5, max_shift)
  test  <- make_curves(50, max_shift)
  accuracy_table(train$M, train$group, test$M, test$group)
})
round(acc_sim, 2)

- ずれがなければ、3つの方法とも、ほぼすべて当たります。
- ずれがあると、DTWを使った最近傍法だけが、正解率を保ちます。

### A-6. 訓練とテストの分け方

クラスタリングと判別は、系列どうしを、互いに独立な観測（iid）として扱います。
1本の長い系列を1日ごとに切って作った曲線では、となりの日どうしが似ています。第2回で見た「隣が見えている」問題が、ここでも起きるかを確かめます。

200日分の曲線を作ります。ラベルは2つで、曲線の違いは小さくしてあります。曲線には、ラベルと関係のない、日ごとの上下（天気のようなもの）が乗っています。

- `iid`：ラベルも、日ごとの上下も、日ごとに独立に決まる。
- `dependent`：ラベルは20日ずつ続く。日ごとの上下は、ゆっくり変わる（AR(1)、$\phi = 0.98$）。

テストデータの50日を、2通りに選びます。`random`はランダムに50日、`block`は最後の50日です。最近傍法（ユークリッド距離）で当てて、100回くり返します。

In [ ]:
set.seed(1)
u <- (1:m - 24.5) / 24
B <- cbind(1, u, u^2 - mean(u^2))                 # 日ごとの上下の形：高さ、傾き、曲がり
ar1_days <- function(n, phi) {                    # 分散1のAR(1)（最初の100個は捨てる）
  as.numeric(stats::filter(rnorm(n + 100, sd = sqrt(1 - phi^2)), phi, method = "recursive"))[-(1:100)]
}
one_split <- function(phi, run, n = 200, n_test = 50) {
  label <- if (run > 1) rep(rep(1:2, length.out = n / run), each = run) else sample(1:2, n, replace = TRUE)
  weather <- sapply(1:3, function(k) ar1_days(n, phi)) %*% t(B)
  M <- weather + outer(as.numeric(label == 2), 0.4 * bump(1:m, 30, 4)) + matrix(rnorm(n * m, sd = 0.1), n)
  acc <- function(test) {
    mean(nearest_label(euclid_cross(M[test, ], M[-test, ]), label[-test]) == label[test])
  }
  c(random = acc(sort(sample(n, n_test))), block = acc((n - n_test + 1):n))
}
res_split <- rbind(iid       = rowMeans(replicate(100, one_split(phi = 0, run = 1))),
                   dependent = rowMeans(replicate(100, one_split(phi = 0.98, run = 20))))
round(res_split, 2)

- `iid`：2つの分け方で、正解率はほぼ同じです。
- `dependent`：ランダムに分けると、正解率がずっと高く出ます。テストの日のとなりの日が訓練データに入っていて、その日は、天気もラベルも同じだからです。

日ごとに切った曲線を判別するときは、テストデータを、訓練データから離れた、ひとまとまりの期間にします。

## B. 実データを解析する

`tsibbledata`パッケージの`vic_elec`を使います。オーストラリアのビクトリア州の、30分ごとの電力需要です。第2回と第12回でも使いました。
1日を1本の曲線（48点）とみて、曲線どうしの「似ている」を測ります。

B節は、A節で作った関数（`dtw_dist`、`ari`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series      <- "Demand"   # 曲線にする系列："Demand"（電力需要）か "Temperature"（気温）
year        <- 2014       # クラスタリングに使い、判別ではテストデータにする年：2012, 2013, 2014 のどれか
scale_curve <- TRUE       # TRUE：曲線ごとに標準化する（形だけを比べる）。FALSE：もとの単位のまま（高さも比べる）
label       <- "work"     # 当てるラベル："work"（平日か休みか）か "season"（季節）
k           <- 2          # クラスターの数：2〜4
window      <- 4          # DTWの窓の幅（30分を1とする）：0〜8

### B-1. データを確かめる

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Date)
per_day <- vic_elec |> as_tibble() |> count(Date, name = "points")   # 1日の点の数
table(per_day$points)
per_day |> filter(points != 48)

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）、`Holiday`が祝日の印です。
- 期間は2012年1月1日から2014年12月31日です。
- 1日の点の数は、48とは限りません。46点の日と50点の日が、3日ずつあります。夏時間の始まり（10月）と終わり（4月）の日です。

### B-2. 前処理：1日ごとの曲線に切る

第12回と同じく、48点がそろっていない6日を除きます。1日を1行、時刻を列にした行列を作ります（1090行、48列）。
あわせて、1日ごとのラベルを2種類作ります。

- `work`：平日（`workday`）か、休み（`dayoff`。土曜、日曜、祝日）か
- `season`：季節。南半球なので、12〜2月が夏（`summer`）、3〜5月が秋（`autumn`）、6〜8月が冬（`winter`）、9〜11月が春（`spring`）

In [ ]:
elec <- vic_elec |>
  as_tibble() |>
  filter(Date %in% per_day$Date[per_day$points == 48]) |>
  arrange(Time)
curves_all <- matrix(elec[[series]], ncol = 48, byrow = TRUE)     # 1行が1日、1列が時刻
if (series == "Demand") curves_all <- curves_all / 1000           # MWh を GWh にする
days <- elec |>
  group_by(Date) |>
  summarise(Holiday = any(Holiday)) |>
  mutate(work = ifelse(Holiday | format(Date, "%u") %in% c("6", "7"), "dayoff", "workday"),
         season = c("summer", "summer", "autumn", "autumn", "autumn", "winter", "winter", "winter",
                    "spring", "spring", "spring", "summer")[as.integer(format(Date, "%m"))])
dim(curves_all)
table(days$work, days$season)

設定に合わせて、曲線を標準化し、1年分を取り出します。

In [ ]:
if (scale_curve) curves_all <- standardise(curves_all)
y_lab <- paste0(if (series == "Demand") "Demand (GWh per 30 min)" else "Temperature (deg C)",
                if (scale_curve) ", standardised" else "")
hour <- (0:47) / 2                                   # 時刻（時）
in_year <- format(days$Date, "%Y") == year
M   <- curves_all[in_year, ]                         # この年の曲線
lab <- days[[label]][in_year]                        # この年のラベル
table(lab)
plot_curves(M, lab, hour) +
  labs(x = "Time of day (hour)", y = y_lab)

### B-3. 4つの距離

この年の曲線どうしの距離を、4通りに計算します。

- `euclid`：ユークリッド距離
- `dtw`：DTW（窓の幅は`window`）
- `feature`：特徴量による距離。1日の曲線の中には季節がないので、季節の強さは使えません。傾向の強さ、ラグ1の自己相関、差分のラグ1の自己相関の3つを使います。
- `fpca`：FPCAのスコア（3つ）による距離

In [ ]:
ft_real  <- series_features(M) |> select(trend_strength, acf1, diff1_acf1)
fit_real <- fpca_fit(M)
D <- list(euclid  = dist(M),
          dtw     = dtw_dist(M, window),
          feature = dist(scale(ft_real)),
          fpca    = dist(fit_real$scores))
round(fit_real$varprop, 2)

### B-4. クラスタリングして、分け方を比べる

4つの距離のそれぞれで、k-medoids法で`k`個に分けます。

In [ ]:
cl <- lapply(D, function(D) cluster::pam(D, k = k)$clustering)
table(label = lab, euclid = cl$euclid)
table(label = lab, feature = cl$feature)

クラスターごとに平均した曲線を、距離ごとに描きます。

In [ ]:
mean_curves <- do.call(rbind, lapply(names(cl), function(nm) {
  do.call(rbind, lapply(1:k, function(g) {
    data.frame(distance = nm, cluster = factor(g), hour = hour,
               value = colMeans(M[cl[[nm]] == g, , drop = FALSE]))
  }))
}))
ggplot(mean_curves, aes(hour, value, colour = cluster)) +
  geom_line() +
  facet_wrap(~distance, nrow = 1) +
  labs(x = "Time of day (hour)", y = paste("Mean of", y_lab), colour = "Cluster")

分け方どうしの一致を、ARIで比べます。`label`の行と列は、ラベルとの一致です。

In [ ]:
parts <- c(cl, list(label = lab))
ari_real <- sapply(parts, function(a) sapply(parts, function(b) ari(a, b)))
round(ari_real, 2)

### B-5. 判別

この年（`year`）をテストデータ、残りの2年を訓練データにして、ラベルを当てます。
比べる基準（ベンチマーク）は、「訓練データでいちばん多いラベルを、いつも答える」方法（`majority`）です。

In [ ]:
M_train   <- curves_all[!in_year, ]
lab_train <- days[[label]][!in_year]
c(train = nrow(M_train), test = nrow(M))
acc <- c(majority = mean(lab == names(which.max(table(lab_train)))),
         accuracy_table(M_train, lab_train, M, lab, window))
round(acc, 3)

どのラベルを、どのラベルと間違えたかを見ます（最近傍法、ユークリッド距離）。

In [ ]:
pred_nn <- nearest_label(euclid_cross(M, M_train), lab_train)
table(label = lab, predicted = pred_nn)

### B-6. 分け方を確かめる

A-6の問題が、このデータで起きるかを確かめます。3年分のすべての日から、テストデータと同じ数の日を、ランダムに選びます。

- `accuracy`：最近傍法（ユークリッド距離）の正解率
- `neighbour_1day`：いちばん近い曲線が、前の日か次の日だった割合
- `neighbour_7days`：いちばん近い曲線が、前後7日以内だった割合

In [ ]:
set.seed(1)
split_check <- function(test) {
  D_cross <- euclid_cross(curves_all[test, ], curves_all[-test, ])
  nearest <- apply(D_cross, 1, which.min)
  gap <- abs(as.numeric(days$Date[-test][nearest] - days$Date[test]))    # いちばん近い曲線との日数の差
  c(accuracy = mean(days[[label]][-test][nearest] == days[[label]][test]),
    neighbour_1day = mean(gap <= 1), neighbour_7days = mean(gap <= 7))
}
acc_split <- rbind(block  = split_check(which(in_year)),
                   random = split_check(sort(sample(nrow(curves_all), sum(in_year)))))
round(acc_split, 3)

読み方：

- B-4：1日の曲線は、時刻がそろっています。A-2の`aligned`に近いので、DTWがユークリッド距離より良くなるとは限りません。
- B-4：特徴量は、「いつ」高いかを捨てて、「どのくらいなめらかか」だけを残します。平日と休みの違いが「朝の立ち上がりの時刻」にあるなら、特徴量では分かれません。
- B-5：正解率は、`majority`と比べて読みます。
- B-6：ランダムに分けたときの正解率が、ひとまとまりの期間で分けたときより高ければ、となりの日が見えていることを疑います。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"Demand"` `"Temperature"` |
| `year` | 2012、2013、2014 |
| `scale_curve` | `TRUE` `FALSE` |
| `label` | `"work"` `"season"` |
| `k` | 2〜4 |
| `window` | 0〜8 |

読むときの観点：

1. `label`を`"season"`、`k`を4にすると、クラスターは季節と一致するか。判別の正解率は、どの方法が高いか。クラスタリング（ラベルを使わない）と判別（ラベルを使う）の違いを考える。
2. `scale_curve`を`FALSE`にすると、クラスターの平均の曲線は、何で分かれるようになるか。
3. `series`を`"Temperature"`にすると、気温の曲線から、平日か休みかを当てられるか。正解率を`majority`と比べる。
4. `window`を0、2、8と変えると、DTWによる分け方とユークリッド距離による分け方の一致（ARI）は、どう変わるか。
5. B-6で、ランダムに分けたときの正解率が高く出る設定はあるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Demand、2014年、標準化する、work、2個、窓の幅4）の結果と比べてください。